In [1]:


#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

from pandas import ExcelWriter

import string

import os

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.alert import Alert

In [3]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'CA FSRA' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running CA FSRA Web Scraping Tool v.1.0


In [4]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()




# %%

In [5]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

         regulatorName + ' 1': 'https://www.fsrao.ca/consumers/credit-unions-and-deposit-insurance/find-credit-union-or-caisses-populaires-ontario?combine=&inactive_effective_date_from=&inactive_effective_date_to=&field_credit_union_status_value=Active',
          regulatorName + ' 2': 'http://licensingcomplaintofficers.fsco.gov.on.ca/LicClass/eng/fraternity_lic_companies_class.aspx',
          regulatorName + ' 3': 'http://licensingcomplaintofficers.fsco.gov.on.ca/LicClass/eng/lic_companies_class.aspx',
         regulatorName + ' 4': 'https://loanandtrust.fsco.gov.on.ca/?name=LoanAndTrust&amp%3Bculture=en-CA',

        }



Typology={

        regulatorName+' 1': 'List of Insured Credit Unions and Caisse Populaires',
        regulatorName+' 2': 'List of Licensed Insurance Companies - Fraternal Societies',
        regulatorName+' 3': 'List of Licensed Insurance Companies',
        regulatorName+' 4': 'List of Licensed Loan or Trust Company',


        }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [],
         'Address_1': [], 'Address_2': [], 'City': [], 'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 
         'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 'Phone - Mother company': [], 'Check': []}



searchValues = list(string.ascii_lowercase) + list(map(str, range(10)))

now = datetime.datetime.now()

processdate = now.strftime('%Y-%m-%d')




In [6]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

def scroll_to_bottom(driver):
    # Get scroll height

    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:
        # Scroll down to the bottom

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        # Wait to load the page

        sleep(2)

        # Calculate new scroll height and compare with last scroll height

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:

            break
        last_height = new_height


In [11]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for reg in regdict:
    print(f'Working with list {reg}')
    driver.get(regdict[reg])
    sleep(5)
    soup = BeautifulSoup(driver.page_source, 'html.parser')  
    if reg == 'CA FSRA 1':
        sleep(2)
        LastPage = True
        while LastPage:
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            pager = soup.find('nav', class_='pager')
            try:
                page_items = pager.find('ul')
                pages_li = page_items.find_all('li', class_='pager__item')
            except:
                LastPage = True
                pages_li = ''
            
            table = soup.find('table', class_='views-table')
            tbody = table.find('tbody')
            trs = tbody.find_all('tr')
            for tr in trs:
                tds = tr.find_all('td')
                name = tds[0].text.strip()
                sqldict['Name'].append(name.strip())
                print(tds[0].text.strip())
                sqldict['ListProcessDate'].append(processdate)
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegCtry'].append(reg.split(' ')[0]) 
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])          
                sqldict['RegulationType'].append('Regulated')       
            if pages_li: 
                page_info = [page.find('span').text.strip() for page in pages_li]

        
                if 'Next page' in page_info:
                    # find li element in page
                    next_button_element = [page for page in pages_li if page.find('span').text.strip() == 'Next page'][0]
                    print('Next page found, navigating...')
                    next_href = next_button_element.find('a')['href']
                    next_http = 'https://www.fsrao.ca/consumers/credit-unions-and-deposit-insurance/find-credit-union-or-caisses-populaires-ontario' + next_href
                    driver.get(next_http)
                    sleep(2)  
            else:
                print('Last Page')
                LastPage = False
        sqldict = bourange_same_length_array(sqldict)  


    elif reg == 'CA FSRA 2' or  reg == 'CA FSRA 3':

        sleep(5)
        allButton = driver.find_element(By.XPATH, '//*[@id="GridView1"]/tbody/tr[1]/th[1]/a[1]')
        driver.execute_script("arguments[0].click();", allButton)
        sleep(5)
        scroll_to_bottom(driver)
        sleep(5)
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        sleep(5)
        table = soup.find('table',id = 'GridView1')
        sleep(5)
        allButton = table.find('tr',class_='Header')
        sleep(5)
        trs = table.find('tbody').find_all('tr')
        sleep(5)
        names =  table.find('tbody').find_all('tr',attrs={"align":"left"})
        sleep(5)
        for name in names:
            tds = name.find_all('td')
            name = tds[1].text
            sqldict['Name'].append(name)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegCtry'].append(reg.split(' ')[0]) 
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])          
            sqldict['RegulationType'].append('Regulated')
        for i,tr in enumerate(trs):
            tds = tr.find_all('td')
            for index,td in enumerate(tds):
                if index == 1:
                    infos = td.find('tbody')
                    if infos != None:
                        if len(infos.find_all('td')) >1:
                        # print(len(infos.find_all('td')))
                            address = infos.find_all('td')[5].text
                            #print("address:",address)
                            address = address.replace('\n',' ')
                            sqldict['Address_1'].append(address)
                            c_p_p = infos.find_all('td')[7].text
                            city = c_p_p.split(',')[0]
                            postcode = c_p_p.split(',')[-1]
                            sqldict['City'].append(city)
                            sqldict['Zip'].append(postcode)
                            print('cpp:',c_p_p)
                            country = infos.find_all('td')[9].text
                            print('country:',country)
                            tele = infos.find_all('td')[11].text
                            print('tele:',tele)
                            if len(tele) >1:
                                sqldict['Phone'].append(tele)
                            else:
                                sqldict['Phone'].append('')
        sqldict = bourange_same_length_array(sqldict)  
    elif reg == 'CA FSRA 4':
        sleep(2)
        table = soup.find('div',id="reactRoot")
        rows = table.find_all(attrs={"role": "row"})
        for row in rows:
            #print(row)
            cells = row.find_all(attrs={"role": "cell"})
            for index,cell in enumerate(cells):
                #print(index, cell.text)
                if index == 0 :
                    if cell.text.strip() == '':
                        pass
                    else:
                        name = cell.text.strip()
                        #print(name)
                        sqldict['Name'].append(name)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListName'].append(Typology[reg])
                        sqldict['RegCtry'].append(reg.split(' ')[0]) 
                        sqldict['RegCode'].append(reg.split(' ')[1])
                        sqldict['ListCode'].append(reg.split(' ')[-1])          
                        sqldict['RegulationType'].append('Regulated')
                elif index == 1:
                    address = cell.text.strip()
                    #print(address)
                    sqldict['Address_1'].append(address)
                elif index == 2:
                    city_prov = cell.text.strip()
                    #print(city_prov)
                    sqldict['City'].append(city_prov)
                elif index == 3:
                    zip = cell.text.strip()
                    #print(zip)
                    sqldict['Zip'].append(zip)
                elif index == 3:
                    phone = cell.text.strip()
                    #print(phone)
                    if len(phone)>1:
                        sqldict['Phone'].append(phone)
        sqldict = bourange_same_length_array(sqldict)  


Working with list CA FSRA 1
Alterna Savings and Credit Union Limited
Bay Credit Union Limited
Buduchnist Credit Union Limited
Caisse Desjardins Ontario Credit Union Inc.
Caisse populaire Alliance limitée
DUCA Financial Services Credit Union Ltd.
Energy Credit Union Limited (The)
Equity Credit Union Inc.
Finnish Credit Union Limited
FirstOntario Credit Union Limited
Fort York Community Credit Union Limited
Frontline Financial Credit Union Limited
Ganaraska Credit Union Ltd.
Golden Horseshoe Credit Union Limited
Healthcare and Municipal Employees' Credit Union Limited
Italian Canadian Savings & Credit Union Limited
Kawartha Credit Union Limited
Kindred Credit Union Limited
Kingston Community Credit Union Limited
Korean (Toronto) Credit Union Limited
Korean Catholic Church Credit Union Limited
L.I.U.N.A. Local 183 Credit Union Limited
Libro Credit Union Limited
Lighthouse Credit Union Limited
Luminus Financial Services & Credit Union Limited
Mainstreet Credit Union Limited
Meridian Credit

In [12]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)



C:\Users\wuj1\AppData\Local\Temp\3\ipykernel_23908\2044201188.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [13]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 526 values.
Key 'priority' has 526 values.
Key 'ListLabel' has 526 values.
Key 'Typology' has 526 values.
Key 'EntryType' has 526 values.
Key 'Name' has 526 values.
Key 'InternalID_1' has 526 values.
Key 'InternalID_1_type' has 526 values.
Key 'InternalID_2' has 526 values.
Key 'InternalID_2_type' has 526 values.
Key 'InternalID_3' has 526 values.
Key 'InternalID_3_type' has 526 values.
Key 'CoType' has 526 values.
Key 'License_Type' has 526 values.
Key 'Address_1' has 526 values.
Key 'Address_2' has 526 values.
Key 'City' has 526 values.
Key 'Zip' has 526 values.
Key 'Cntry' has 526 values.
Key 'Phone' has 526 values.
Key 'Fax' has 526 values.
Key 'Website' has 526 values.
Key 'Email' has 526 values.
Key 'RegulationType' has 526 values.
Key 'RegulationTypeCode' has 526 values.
Key 'RegulationDate' has 526 values.
Key 'CancellationDate' has 526 values.
Key 'RegCtry' has 526 values.
Key 'RegCode' has 526 values.
Key 'ListCode' has 526 values.
Key 'ListLanguage' has 526 v

In [14]:
df.to_csv('TOTAL_v1.csv')